# **Space X  Falcon 9 First Stage Landing Prediction**


## Web scraping Falcon 9 and Falcon Heavy Launches Records from Wikipedia


In this lab, you will be performing web scraping to collect Falcon 9 historical launch records from a Wikipedia page titled `List of Falcon 9 and Falcon Heavy launches`

https://en.wikipedia.org/wiki/List_of_Falcon_9_and_Falcon_Heavy_launches


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_1_L2/images/Falcon9_rocket_family.svg)


Falcon 9 first stage will land successfully


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DS0701EN-SkillsNetwork/api/Images/landing_1.gif)


Several examples of an unsuccessful landing are shown here:


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DS0701EN-SkillsNetwork/api/Images/crash.gif)


More specifically, the launch records are stored in a HTML table shown below:


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_1_L2/images/falcon9-launches-wiki.png)


  ## Objectives
Web scrap Falcon 9 launch records with `BeautifulSoup`: 
- Extract a Falcon 9 launch records HTML table from Wikipedia
- Parse the table and convert it into a Pandas data frame


First let's import required packages for this lab


In [7]:
!pip install beautifulsoup4
!pip install requests
!pip install pandas
!pip install numpy

In [8]:
import sys

import requests
from bs4 import BeautifulSoup
import re
import unicodedata
import pandas as pd

and we will provide some helper functions for you to process web scraped HTML table


In [9]:
def date_time(table_cells):
    """
    This function returns the data and time from the HTML  table cell
    Input: the  element of a table data cell extracts extra row
    """
    return [data_time.strip() for data_time in list(table_cells.strings)][0:2]

def booster_version(table_cells):
    """
    This function returns the booster version from the HTML  table cell 
    Input: the  element of a table data cell extracts extra row
    """
    out=''.join([booster_version for i,booster_version in enumerate( table_cells.strings) if i%2==0][0:-1])
    return out

def landing_status(table_cells):
    """
    This function returns the landing status from the HTML table cell 
    Input: the  element of a table data cell extracts extra row
    """
    out=[i for i in table_cells.strings][0]
    return out


def get_mass(table_cells):
    mass=unicodedata.normalize("NFKD", table_cells.text).strip()
    if mass:
        mass.find("kg")
        new_mass=mass[0:mass.find("kg")+2]
    else:
        new_mass=0
    return new_mass


def extract_column_from_header(row):
    """
    This function returns the landing status from the HTML table cell 
    Input: the  element of a table data cell extracts extra row
    """
    if (row.br):
        row.br.extract()
    if row.a:
        row.a.extract()
    if row.sup:
        row.sup.extract()
        
    colunm_name = ' '.join(row.contents)
    
    # Filter the digit and empty names
    if not(colunm_name.strip().isdigit()):
        colunm_name = colunm_name.strip()
        return colunm_name    


To keep the lab tasks consistent, you will be asked to scrape the data from a snapshot of the  `List of Falcon 9 and Falcon Heavy launches` Wikipage updated on
`9th June 2021`


In [10]:
static_url = "https://en.wikipedia.org/w/index.php?title=List_of_Falcon_9_and_Falcon_Heavy_launches&oldid=1027686922"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/91.0.4472.124 Safari/537.36"
}

Next, request the HTML page from the above URL and get a `response` object


### TASK 1: Request the Falcon9 Launch Wiki page from its URL


First, let's perform an HTTP GET method to request the Falcon9 Launch HTML page, as an HTTP response.


In [11]:
# use requests.get() method with the provided static_url and headers
# assign the response to a object
# Perform HTTP GET request
response = requests.get(static_url, headers=headers)

# Check status
print("Status Code:", response.status_code)

# Preview first 500 characters of HTML
print(response.text[:500])


Status Code: 200
<!DOCTYPE html>
<html class="client-nojs vector-feature-language-in-header-enabled vector-feature-language-in-main-page-header-disabled vector-feature-page-tools-pinned-disabled vector-feature-toc-pinned-clientpref-1 vector-feature-main-menu-pinned-disabled vector-feature-limited-width-clientpref-1 vector-feature-limited-width-content-enabled vector-feature-custom-font-size-clientpref-1 vector-feature-appearance-pinned-clientpref-1 skin-theme-clientpref-day vector-sticky-header-enabled wp25easte


Create a `BeautifulSoup` object from the HTML `response`


In [12]:
# Use BeautifulSoup() to create a BeautifulSoup object from a response text content
# Create BeautifulSoup object from the HTML response
soup = BeautifulSoup(response.text, "html.parser")

# Quick check: print the page title
print(soup.title.string)


List of Falcon 9 and Falcon Heavy launches - Wikipedia


Print the page title to verify if the `BeautifulSoup` object was created properly 


In [15]:
# Use soup.title attribute
# Verify BeautifulSoup object by printing the page title
print(soup.title.string)



List of Falcon 9 and Falcon Heavy launches - Wikipedia


### TASK 2: Extract all column/variable names from the HTML table header


Next, we want to collect all relevant column names from the HTML table header


Let's try to find all tables on the wiki page first. If you need to refresh your memory about `BeautifulSoup`, please check the external reference link towards the end of this lab


In [16]:
# Use the find_all function in the BeautifulSoup object, with element type `table`
# Assign the result to a list called `html_tables`
# Find all tables on the page
html_tables = soup.find_all("table")

# Quick check: how many tables were found?
print("Number of tables found:", len(html_tables))

# Preview the first table headers
first_table = html_tables[0]
headers = [th.get_text(strip=True) for th in first_table.find_all("th")]
print("Column names:", headers)


Number of tables found: 25
Column names: []


Starting from the third table is our target table contains the actual launch records.


In [17]:
# Let's print the third table and check its content
first_launch_table = html_tables[2]
print(first_launch_table)

<table class="wikitable plainrowheaders collapsible" style="width: 100%;">
<tbody><tr>
<th scope="col">Flight No.
</th>
<th scope="col">Date and<br/>time (<a href="/wiki/Coordinated_Universal_Time" title="Coordinated Universal Time">UTC</a>)
</th>
<th scope="col"><a href="/wiki/List_of_Falcon_9_first-stage_boosters" title="List of Falcon 9 first-stage boosters">Version,<br/>Booster</a> <sup class="reference" id="cite_ref-booster_11-0"><a href="#cite_note-booster-11"><span class="cite-bracket">[</span>b<span class="cite-bracket">]</span></a></sup>
</th>
<th scope="col">Launch site
</th>
<th scope="col">Payload<sup class="reference" id="cite_ref-Dragon_12-0"><a href="#cite_note-Dragon-12"><span class="cite-bracket">[</span>c<span class="cite-bracket">]</span></a></sup>
</th>
<th scope="col">Payload mass
</th>
<th scope="col">Orbit
</th>
<th scope="col">Customer
</th>
<th scope="col">Launch<br/>outcome
</th>
<th scope="col"><a href="/wiki/Falcon_9_first-stage_landing_tests" title="Falcon 

You should able to see the columns names embedded in the table header elements `<th>` as follows:


```
<tr>
<th scope="col">Flight No.
</th>
<th scope="col">Date and<br/>time (<a href="/wiki/Coordinated_Universal_Time" title="Coordinated Universal Time">UTC</a>)
</th>
<th scope="col"><a href="/wiki/List_of_Falcon_9_first-stage_boosters" title="List of Falcon 9 first-stage boosters">Version,<br/>Booster</a> <sup class="reference" id="cite_ref-booster_11-0"><a href="#cite_note-booster-11">[b]</a></sup>
</th>
<th scope="col">Launch site
</th>
<th scope="col">Payload<sup class="reference" id="cite_ref-Dragon_12-0"><a href="#cite_note-Dragon-12">[c]</a></sup>
</th>
<th scope="col">Payload mass
</th>
<th scope="col">Orbit
</th>
<th scope="col">Customer
</th>
<th scope="col">Launch<br/>outcome
</th>
<th scope="col"><a href="/wiki/Falcon_9_first-stage_landing_tests" title="Falcon 9 first-stage landing tests">Booster<br/>landing</a>
</th></tr>
```


Next, we just need to iterate through the `<th>` elements and apply the provided `extract_column_from_header()` to extract column name one by one


In [18]:
column_names = []

# Apply find_all() function with `th` element on first_launch_table
# Iterate each th element and apply the provided extract_column_from_header() to get a column name
# Append the Non-empty column name (`if name is not None and len(name) > 0`) into a list called column_names

# Loop through all tables until we find one with "Flight No." in its headers
first_launch_table = None
for tbl in html_tables:
    headers = [th.get_text(strip=True) for th in tbl.find_all("th")]
    if "Flight No." in headers:
        first_launch_table = tbl
        break

# Now extract clean column names using your helper
column_names = []
for th in first_launch_table.find_all("th"):
    name = extract_column_from_header(th)
    if name is not None and len(name) > 0:
        column_names.append(name)

print("Extracted column names:", column_names)

Extracted column names: ['Flight No.', 'Date and time ( )', 'Launch site', 'Payload', 'Payload mass', 'Orbit', 'Customer', 'Launch outcome']


Check the extracted column names


In [19]:
print(column_names)
    
# Look at the first table directly
first_launch_table = html_tables[0]

# Print raw header texts without cleaning
raw_headers = [th.get_text(strip=True) for th in first_launch_table.find_all("th")]
print("Raw headers:", raw_headers)

column_names = []
for th in first_launch_table.find_all("th"):
    name = th.get_text(strip=True)
    if name:
        column_names.append(name)

print("Extracted column names:", column_names)

first_launch_table = html_tables[1]  # or another index

['Flight No.', 'Date and time ( )', 'Launch site', 'Payload', 'Payload mass', 'Orbit', 'Customer', 'Launch outcome']
Raw headers: []
Extracted column names: []


In [20]:
# Loop through all tables and print their first few header cells
for idx, tbl in enumerate(html_tables):
    headers = [th.get_text(strip=True) for th in tbl.find_all("th")]
    print(f"Table {idx} headers:", headers[:5])  # show first 5 headers

Table 0 headers: []
Table 1 headers: []
Table 2 headers: ['Flight No.', 'Date andtime ()', '', 'Launch site', 'Payload']
Table 3 headers: ['Flight No.', 'Date andtime (UTC)', 'Version,Booster[b]', 'Launch site', 'Payload[c]']
Table 4 headers: ['Flight No.', 'Date andtime (UTC)', 'Version,Booster[b]', 'Launch site', 'Payload[c]']
Table 5 headers: ['Flight No.', 'Date andtime (UTC)', 'Version,Booster[b]', 'Launch site', 'Payload[c]']
Table 6 headers: ['Flight No.', 'Date andtime (UTC)', 'Version,Booster[b]', 'Launch site', 'Payload[c]']
Table 7 headers: ['Flight No.', 'Date andtime (UTC)', 'Version,Booster[b]', 'Launch site', 'Payload[c]']
Table 8 headers: ['Flight No.', 'Date andtime (UTC)', 'Version,Booster[b]', 'Launchsite', 'Payload[c]']
Table 9 headers: ['Flight No.', 'Date andtime (UTC)', 'Version,Booster[b]', 'Launchsite', 'Payload[c]']
Table 10 headers: ['FlightNo.', 'Date andtime (UTC)', 'Version,Booster[b]', 'Launchsite', 'Payload[c]']
Table 11 headers: ['Date and time (UTC)', 

## TASK 3: Create a data frame by parsing the launch HTML tables


We will create an empty dictionary with keys from the extracted column names in the previous task. Later, this dictionary will be converted into a Pandas dataframe


In [21]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import unicodedata

# --- Snapshot URL (June 9, 2021) ---
static_url = "https://en.wikipedia.org/w/index.php?title=List_of_Falcon_9_and_Falcon_Heavy_launches&oldid=1027686922"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/91.0.4472.124 Safari/537.36"
}

# --- Helper functions ---
def date_time(table_cells):
    return [s.strip() for s in list(table_cells.strings)][0:2]

def booster_version(table_cells):
    out = ''.join([text for i, text in enumerate(table_cells.strings) if i % 2 == 0][:-1])
    return out.strip()

def landing_status(table_cells):
    return next(table_cells.strings).strip()

def get_mass(table_cells):
    mass = unicodedata.normalize("NFKD", table_cells.text).strip()
    if "kg" in mass:
        return mass[:mass.find("kg")+2]
    return "0"

def extract_column_from_header(row):
    for tag in ["br", "a", "sup"]:
        if getattr(row, tag):
            getattr(row, tag).decompose()
    col_name = ' '.join(row.stripped_strings)
    if not col_name.isdigit():
        return col_name.strip()

# --- Step 1: Request page ---
response = requests.get(static_url, headers=headers)
soup = BeautifulSoup(response.text, "html.parser")

print("Page title:", soup.title.string)

# --- Step 2: Find all tables ---
html_tables = soup.find_all("table", {"class": "wikitable"})

# --- Step 3: Identify Falcon 9 table (loop until we find headers) ---
for idx, tbl in enumerate(html_tables):
    headers = [th.get_text(strip=True) for th in tbl.find_all("th")]
    if "Flight No." in headers:
        first_launch_table = tbl
        break

# --- Step 4: Extract column names ---
column_names = []
for th in first_launch_table.find_all("th"):
    name = extract_column_from_header(th)
    if name is not None and len(name) > 0:
        column_names.append(name)

print("Extracted column names:", column_names)

# --- Step 5: Initialize dictionary ---
launch_dict = {
    'Flight No.': [],
    'Launch site': [],
    'Payload': [],
    'Payload mass': [],
    'Orbit': [],
    'Customer': [],
    'Launch outcome': [],
    'Version Booster': [],
    'Booster landing': [],
    'Date': [],
    'Time': []
}

# --- Step 6: Parse rows ---
for tr in first_launch_table.find_all("tr")[1:]:
    cells = tr.find_all("td")
    if not cells:
        continue
    
    try:
        flight_no = cells[0].get_text(strip=True)
        date, time = date_time(cells[1])
        booster = booster_version(cells[2])
        launch_site = cells[3].get_text(strip=True)
        payload = cells[4].get_text(strip=True)
        payload_mass = get_mass(cells[5])
        orbit = cells[6].get_text(strip=True)
        customer = cells[7].get_text(strip=True)
        launch_outcome = cells[8].get_text(strip=True)
        booster_landing = landing_status(cells[9])
    except Exception as e:
        continue
    
    # Append to dictionary
    launch_dict['Flight No.'].append(flight_no)
    launch_dict['Date'].append(date)
    launch_dict['Time'].append(time)
    launch_dict['Version Booster'].append(booster)
    launch_dict['Launch site'].append(launch_site)
    launch_dict['Payload'].append(payload)
    launch_dict['Payload mass'].append(payload_mass)
    launch_dict['Orbit'].append(orbit)
    launch_dict['Customer'].append(customer)
    launch_dict['Launch outcome'].append(launch_outcome)
    launch_dict['Booster landing'].append(booster_landing)

# --- Step 7: Convert to DataFrame ---
launch_df = pd.DataFrame(launch_dict)

print(launch_df.head())

Page title: List of Falcon 9 and Falcon Heavy launches - Wikipedia
Extracted column names: ['Flight No.', 'Date and time ( )', 'Launch site', 'Payload', 'Payload mass', 'Orbit', 'Customer', 'Launch outcome']
Empty DataFrame
Columns: [Flight No., Launch site, Payload, Payload mass, Orbit, Customer, Launch outcome, Version Booster, Booster landing, Date, Time]
Index: []


In [22]:
launch_dict= dict.fromkeys(column_names)

# Remove an irrelvant column
del launch_dict['Date and time ( )']

# Let's initial the launch_dict with each value to be an empty list
launch_dict['Flight No.'] = []
launch_dict['Launch site'] = []
launch_dict['Payload'] = []
launch_dict['Payload mass'] = []
launch_dict['Orbit'] = []
launch_dict['Customer'] = []
launch_dict['Launch outcome'] = []
# Added some new columns
launch_dict['Version Booster']=[]
launch_dict['Booster landing']=[]
launch_dict['Date']=[]
launch_dict['Time']=[]

Next, we just need to fill up the `launch_dict` with launch records extracted from table rows.


Usually, HTML tables in Wiki pages are likely to contain unexpected annotations and other types of noises, such as reference links `B0004.1[8]`, missing values `N/A [e]`, inconsistent formatting, etc.


To simplify the parsing process, we have provided an incomplete code snippet below to help you to fill up the `launch_dict`. Please complete the following code snippet with TODOs or you can choose to write your own logic to parse all launch tables:


In [24]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import unicodedata

# --- Snapshot URL (June 9, 2021) ---
static_url = "https://en.wikipedia.org/w/index.php?title=List_of_Falcon_9_and_Falcon_Heavy_launches&oldid=1027686922"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/91.0.4472.124 Safari/537.36"
}

# --- Helper functions ---
def date_time(table_cells):
    return [s.strip() for s in list(table_cells.strings)][0:2]

def booster_version(table_cells):
    out = ''.join([text for i, text in enumerate(table_cells.strings) if i % 2 == 0][:-1])
    return out.strip()

def landing_status(table_cells):
    return next(table_cells.strings).strip()

def get_mass(table_cells):
    mass = unicodedata.normalize("NFKD", table_cells.text).strip()
    if "kg" in mass:
        return mass[:mass.find("kg")+2]
    return "0"

def safe_cell_text(cell):
    """Return text from a cell, preferring <a> if present, else plain text."""
    if cell.a and cell.a.string:
        return cell.a.string.strip()
    return cell.get_text(strip=True)

# --- Step 1: Request page ---
response = requests.get(static_url, headers=headers)
soup = BeautifulSoup(response.text, "html.parser")

print("Page title:", soup.title.string)

# --- Step 2: Initialize dictionary ---
launch_dict = {
    'Flight No.': [],
    'Date': [],
    'Time': [],
    'Version Booster': [],
    'Launch site': [],
    'Payload': [],
    'Payload mass': [],
    'Orbit': [],
    'Customer': [],
    'Launch outcome': [],
    'Booster landing': []
}

# --- Step 3: Parse all Falcon 9 tables ---
extracted_row = 0
for table_number, table in enumerate(soup.find_all('table', "wikitable plainrowheaders collapsible")):
    for rows in table.find_all("tr"):
        if rows.th and rows.th.string and rows.th.string.strip().isdigit():
            flight_number = rows.th.string.strip()
            row = rows.find_all('td')

            datatimelist = date_time(row[0])
            date = datatimelist[0].strip(',')
            time = datatimelist[1]

            bv = booster_version(row[1]) or safe_cell_text(row[1])
            launch_site = safe_cell_text(row[2])
            payload = safe_cell_text(row[3])
            payload_mass = get_mass(row[4])
            orbit = safe_cell_text(row[5])
            customer = safe_cell_text(row[6])
            launch_outcome = list(row[7].strings)[0].strip()
            booster_landing = landing_status(row[8])

            # Append into dictionary
            launch_dict['Flight No.'].append(flight_number)
            launch_dict['Date'].append(date)
            launch_dict['Time'].append(time)
            launch_dict['Version Booster'].append(bv)
            launch_dict['Launch site'].append(launch_site)
            launch_dict['Payload'].append(payload)
            launch_dict['Payload mass'].append(payload_mass)
            launch_dict['Orbit'].append(orbit)
            launch_dict['Customer'].append(customer)
            launch_dict['Launch outcome'].append(launch_outcome)
            launch_dict['Booster landing'].append(booster_landing)

            extracted_row += 1

print("Total rows extracted:", extracted_row)

# --- Step 4: Convert to DataFrame ---
launch_df = pd.DataFrame(launch_dict)

print(launch_df.head())

Page title: List of Falcon 9 and Falcon Heavy launches - Wikipedia
Total rows extracted: 121
  Flight No.             Date   Time   Version Booster Launch site  \
0          1      4 June 2010  18:45  F9 v1.07B0003.18       CCAFS   
1          2  8 December 2010  15:43  F9 v1.07B0004.18       CCAFS   
2          3      22 May 2012  07:44  F9 v1.07B0005.18       CCAFS   
3          4   8 October 2012  00:35  F9 v1.07B0006.18       CCAFS   
4          5     1 March 2013  15:10  F9 v1.07B0007.18       CCAFS   

                                Payload Payload mass Orbit Customer  \
0  Dragon Spacecraft Qualification Unit            0   LEO   SpaceX   
1                                Dragon            0   LEO     NASA   
2                                Dragon       525 kg   LEO     NASA   
3                          SpaceX CRS-1     4,700 kg   LEO     NASA   
4                          SpaceX CRS-2     4,877 kg   LEO     NASA   

  Launch outcome Booster landing  
0        Success        

After you have fill in the parsed launch record values into `launch_dict`, you can create a dataframe from it.


In [25]:
df= pd.DataFrame({ key:pd.Series(value) for key, value in launch_dict.items() })

In [26]:
# Export to CSV
df.to_csv("spacex_web_scraped.csv", index=False)

print("CSV file 'spacex_web_scraped.csv' has been created successfully!")


CSV file 'spacex_web_scraped.csv' has been created successfully!


We can now export it to a <b>CSV</b> for the next section, but to make the answers consistent and in case you have difficulties finishing this lab. 

Following labs will be using a provided dataset to make each lab independent. 


<code>df.to_csv('spacex_web_scraped.csv', index=False)</code>
